# Markov Melody Machine 🎵
Teach a **Markov chain** a few children's songs and let it compose new tunes, then *listen* to them (the notebook synthesises a WAV with NumPy). Order 1 sounds random; order 2–3 sounds like a real song; high orders just copy. It's the same trade-off as n-gram language models, in sound.

Background: [[Probability for ML]], [[NLP Overview]] (n-grams) · overview: [[Fun Projects]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import collections, pathlib, wave, numpy as np
from IPython.display import Audio, display
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
# MIDI note numbers (60 = middle C), one entry per beat; public-domain melodies
SONGS = {"Twinkle Twinkle":   [60, 60, 67, 67, 69, 69, 67, 65, 65, 64, 64, 62, 62, 60, 67, 67, 65, 65, 64, 64, 62, 67, 67, 65, 65, 64, 64, 62, 60, 60, 67, 67, 69, 69, 67, 65, 65, 64, 64, 62, 62, 60],
         "Ode to Joy":        [64, 64, 65, 67, 67, 65, 64, 62, 60, 60, 62, 64, 64, 62, 62, 64, 64, 65, 67, 67, 65, 64, 62, 60, 60, 62, 64, 62, 60, 60],
         "Frère Jacques":     [60, 62, 64, 60, 60, 62, 64, 60, 64, 65, 67, 64, 65, 67, 67, 69, 67, 65, 64, 60, 67, 69, 67, 65, 64, 60, 60, 55, 60, 60, 55, 60],
         "Mary Had a Lamb":   [64, 62, 60, 62, 64, 64, 64, 62, 62, 62, 64, 67, 67, 64, 62, 60, 62, 64, 64, 64, 64, 62, 62, 64, 62, 60]}
def synth(notes, path, beat=0.3, sr=22050):
    t = np.linspace(0, beat, int(sr * beat), endpoint=False); env = np.minimum(1, 20 * t) * np.exp(-3 * t)
    audio = np.concatenate([np.sin(2 * np.pi * 440 * 2 ** ((n - 69) / 12) * t) * env for n in notes])
    pathlib.Path(path).parent.mkdir(exist_ok=True)
    with wave.open(str(path), "wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(sr); w.writeframes((audio * 0.5 * 32767).astype(np.int16).tobytes())
    return path

## 1. Learn transitions
For an order-`k` chain, count for every context of `k` consecutive notes which note follows. Return `{context_tuple: Counter(next_note → count)}` over all songs.

In [ ]:
def transitions(songs, order):
    # TODO 1: context → Counter of next notes
    raise NotImplementedError  # TODO 1

## 2. Compose
Start from `start` (a list of `order` notes) and repeatedly sample the next note in proportion to the counts for the current context, until `length` notes (or an unseen context).

In [ ]:
def generate(T, start, order, length=32, seed=0):
    # TODO 2: sample from the transition counts
    raise NotImplementedError  # TODO 2

In [ ]:
def _t_mk():
    T = transitions([[1, 2, 1, 3]], 1); g = generate({(1,): collections.Counter({2: 1})} | {(2,): collections.Counter({1: 1})}, [1], 1, length=5)
    return T == {(1,): collections.Counter({2: 1, 3: 1}), (2,): collections.Counter({1: 1})} and g == [1, 2, 1, 2, 1]
check("markov", _t_mk, "[1,2,1,3] → after 1 comes 2 or 3, after 2 comes 1; a deterministic chain must alternate.")
if ready("markov"):
    songs = list(SONGS.values()); train_ngrams = lambda k: {tuple(s[i:i + 8]) for s in songs for i in range(len(s) - 7)}
    for order in (1, 2, 4):
        T = transitions(songs, order); tune = generate(T, songs[0][:order], order, length=32, seed=3)
        copied = sum(tuple(tune[i:i + 8]) in train_ngrams(8) for i in range(len(tune) - 7)) / max(1, len(tune) - 7)
        print(f"order {order}: {len(tune)} notes, {copied:.0%} of 8-note stretches copied from the songs: {tune[:16]}…")
        display(Audio(synth(tune, f"data/melody_order{order}.wav")))
    T2 = transitions(songs, 2); tune2 = generate(T2, [60, 60], 2, length=40, seed=1)
    check("valid melody", lambda: all(tuple(tune2[i:i + 2]) in T2 and tune2[i + 2] in T2[tuple(tune2[i:i + 2])] for i in range(len(tune2) - 2)) and len(tune2) > 10,
          "Every generated note must follow a context seen in training.")

<details><summary>▶ Solution</summary>

```python
def transitions(songs, order):
    T = {}
    for song in songs:
        for i in range(len(song) - order):
            ctx = tuple(song[i:i + order])
            T.setdefault(ctx, collections.Counter())[song[i + order]] += 1
    return T
```

```python
def generate(T, start, order, length=32, seed=0):
    rng = np.random.default_rng(seed)
    out = list(start)
    while len(out) < length:
        ctx = tuple(out[-order:])
        if ctx not in T:
            break
        notes, counts = zip(*T[ctx].items())
        out.append(int(rng.choice(notes, p=np.array(counts) / sum(counts))))
    return out
```
</details>

## Stretch goals
- Add note *durations* (pairs of pitch and length) so rhythms vary too.
- Transpose every song to all 12 keys before training (data augmentation).
- Add Danish songs you know (e.g. *Jeg har en lille ged*) by writing their MIDI numbers.